# v44 — готовый T01 с Mac mini против рабочего v25

Run All в основном проекте на этом Mac. Это извлечение признаков и сравнение, **не обучение**.

Зафиксированы T01 / step2883 / rung10 и смешивание 90% v25 + 10% T01 перед графом (k1=20, k2=3, λ=0,5). Кандидаты, confidence и порог 0,534365177154541 берутся из прежней R1-ветки v25.

Проверяются только v25 и один заранее выбранный кандидат на исходной validation (309 query, 896 gallery). Эта validation уже наблюдалась раньше и не является независимым скрытым test. Никаких изменений MVP, подбора параметров, скачиваний или остановки обучения Mac mini.


In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ['ORT_DISABLE_TELEMETRY'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '0'
os.environ['PYTORCH_MPS_FAST_MATH'] = '0'
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'training/transreid_t01_validation.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook в основном Car-classification-MSK')
if sys.prefix == sys.base_prefix:
    raise RuntimeError('Выбери kernel проектной .venv, не System Python')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_NAME = 'validation_v1'
DEVICE = 'mps'
print('Python:', sys.executable, '\nПроект:', ROOT, '\nRun:', RUN_NAME, '\nDevice:', DEVICE)
print('Копия v41:', ROOT.parent / 'mac_m4_v1')


In [ ]:
print('ЭТАП 1/3 — проверки scorer, отказов, экспорта и независимости query')
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_transreid_t01_validation.py', 'tests/test_osnet_r1_validation.py'], check=True)


In [ ]:
print('ЭТАП 2/3 — проверка переноса T01, свежие признаки обеих систем, сравнение')
from training.transreid_t01_validation import run, VARIANT
results = run(RUN_NAME, DEVICE)


In [ ]:
from IPython.display import Markdown, display
print('ЭТАП 3/3 — итоговый отчёт')
display(Markdown((VARIANT / 'runs' / RUN_NAME / 'REPORT.md').read_text()))
print('Архив:', VARIANT / f'{RUN_NAME}_analysis.zip')
print('MVP не изменён. Повторный запуск использует проверенный кэш, а не новый замер.')


## Входы и продолжение

Нужны соседние папки `mac_m4_v1` (снимок через AirDrop) и `Car-classification-MSK-main`, а в основном проекте — dataset, исторические v24/v25 и research_transfer/v41_inputs/inputs.json. NiVe-изображения, resume-файлы и исходные pretrained-веса DeiT не используются. Переданную папку не объединять с результатами работающего Mac mini.

При прерывании: Restart Kernel → Run All с тем же RUN_NAME. Завершённые этапы атомарны и проверяются по SHA-256. Изменившиеся источники/веса дают ошибку; допуски и пороги не увеличивать. Свободное место ≥1 GiB, ограничения времени нет.

Кандидат использует 5 энкодеров вместо 4, вектор 2432 float32 (прежние 2048 + T01 384). ONNX-экспорт, GPU/скорость и внедрение в продукт — отдельный этап, автоматического продвижения нет. Аналитический ZIP не содержит веса и NPY; полные CSV/NPY сохраняются в runs/validation_v1.
